In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./pdfs/WHO_ipc.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

/tmp/ipykernel_179075/166776017.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/workspace/learning/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages: 4


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 23


In [3]:
for i, chunk in enumerate(chunks):
    print("=" * 80)
    print("CHUNK:", i)
    print(chunk.page_content)
    print("METADATA:", chunk.metadata)

CHUNK: 0
-1- 
   
 WHO Policy Brief: Maintaining infection 
prevention and control measures for 
COVID-19 in health care facilities  
 
 14 September 2022  
 
    
Key points 
• Health care facilities remain a high-risk SARS-CoV-2 transmission setting because they are 
locations where patients at risk of severe COVID-19 are admitted and cared for.  
• Maintaining and improving infection and control (IPC) measures remain critical for patients, staff 
and visitors.  
• Current key infection prevention and control (IPC) strategies and measures for management of 
COVID-19 in healthcare facilities include1:  
− an IPC programme or at least a dedicated and trained IPC focal point 
− screening and triage for early recognition of community- and health care facility-acquired
METADATA: {'producer': 'Adobe PDF Library 22.2.244', 'creator': 'Acrobat PDFMaker 22 for Word', 'creationdate': '2022-09-12T16:18:58+02:00', 'author': 'Judith Ann MANDELBAUM-SCHMID', 'comments': '', 'company': '', 'keywords

In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(texts)

print(embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2443.43it/s]


(23, 384)


In [5]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="who_ipc"
)

In [6]:
ids = [
    f"who_chunk_{i}"
    for i in range(len(chunks))
]

metadatas = [
    chunk.metadata
    for chunk in chunks
]

embeddings = embeddings.tolist()

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas,
)

print("Stored", len(chunks), "chunks")

Stored 23 chunks


In [7]:
query = "What infection prevention measures are recommended?"

In [8]:
query_embedding = embedding_model.encode(
    query
).tolist()

In [9]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)

In [10]:
for i, document in enumerate(results["documents"][0]):
    print("=" * 80)
    print("RESULT:", i)
    print(document)
    print("PAGE:", results["metadatas"][0][i]["page"])
    print("DISTANCE:", results["distances"][0][i])

RESULT: 0
The policy briefs outline essential actions that national and sub-national policy makers can implement for 
the following: COVID-19 testing, clinical management of COVID-19, reaching COVID-19 vaccination targets, 
maintaining infection prevention and control (IPC) measures for COVID-19 in health care facilities, building 
trust through risk communication and community engagement and managing the COVID -19 infodemic. 
This policy brief focuses on maintaining infection prevention and control (IPC) measures for COVID -19 in 
health care facilities (link to the six policy briefs) . A longer and more detailed policy brief on IPC,  
Maintaining infection prevention and control measures for COVID-19 in health care facilities , was 
published on 7 June 2022. 
Purpose of this document
PAGE: 1
DISTANCE: 0.8081631660461426
RESULT: 1
-1- 
   
 WHO Policy Brief: Maintaining infection 
prevention and control measures for 
COVID-19 in health care facilities  
 
 14 September 2022  
 
    
K

In [18]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
    thinking=False,
)

In [12]:
def retrieve(query: str, n_results: int = 3):
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )

    return results["documents"][0]

In [13]:
docs = retrieve(
    "What infection prevention measures are recommended?"
)

for doc in docs:
    print("=" * 80)
    print(doc)

The policy briefs outline essential actions that national and sub-national policy makers can implement for 
the following: COVID-19 testing, clinical management of COVID-19, reaching COVID-19 vaccination targets, 
maintaining infection prevention and control (IPC) measures for COVID-19 in health care facilities, building 
trust through risk communication and community engagement and managing the COVID -19 infodemic. 
This policy brief focuses on maintaining infection prevention and control (IPC) measures for COVID -19 in 
health care facilities (link to the six policy briefs) . A longer and more detailed policy brief on IPC,  
Maintaining infection prevention and control measures for COVID-19 in health care facilities , was 
published on 7 June 2022. 
Purpose of this document
-1- 
   
 WHO Policy Brief: Maintaining infection 
prevention and control measures for 
COVID-19 in health care facilities  
 
 14 September 2022  
 
    
Key points 
• Health care facilities remain a high-risk SA

In [14]:
context = "\n\n".join(docs)

In [15]:
question = "What infection prevention measures are recommended?"
prompt = f"""
Answer the question using ONLY the information provided in the context.

Context:
{context}

Question:
{question}

If the answer is not present in the context, say:
"I don't know based on the provided document."
"""

In [19]:
response = llm.invoke(prompt)

print(response.content)

Based on the provided context, the recommended infection prevention and control (IPC) measures for managing COVID-19 in healthcare facilities include:

*   Implementing an IPC programme or having at least a dedicated and trained IPC focal point.
*   Conducting screening and triage for the early recognition of community- and health care facility-acquired infections.
*   Regularly undertaking Water, Sanitation, and Hygiene (WASH) measures as required by the International Health Regulations (IHR 2005).
*   Maintaining operational readiness for surges of COVID-19 cases and other emerging or re-emerging pathogens.
